# 🦷 Synthetic Faults Viewer (Colab-Compatible)

This interactive notebook allows you to quickly browse and visualize the synthetic fault meshes and segmentations directly within Google Colab.

In [ ]:
!pip install trimesh scipy plotly ipywidgets

In [ ]:
import os
import json
import trimesh
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Colab-specific initialization for widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Set your synthetic dataset directory here (update path if using Google Drive)
DATASET_ROOT = "synthetic_data"

def get_available_scans(root_dir):
    root_path = Path(root_dir)
    scans = []
    if not root_path.exists():
        return scans
        
    for obj_file in root_path.rglob("*.obj"):
        expected_json = obj_file.parent / f"{obj_file.stem}.json"
        if expected_json.exists():
            json_file = expected_json
        else:
            jsons = [f for f in obj_file.parent.glob("*.json") if "__kpt" not in f.name]
            if not jsons:
                continue
            json_file = jsons[0]
            
        patient_id = obj_file.parent.name
        jaw = 'lower' if 'lower' in obj_file.name.lower() else 'upper'
        
        parts = obj_file.parts
        fault_type = "unknown"
        if len(parts) >= 4:
            fault_type = parts[-4]
            
        display_name = f"{patient_id} - {jaw.capitalize()} ({fault_type.capitalize()})"
        scans.append((display_name, {'obj': str(obj_file), 'json': str(json_file)}))
        
    return sorted(scans, key=lambda x: x[0])

# --- GUI Components ---
scans = get_available_scans(DATASET_ROOT)
if not scans:
    print(f"No data found in {DATASET_ROOT}. Please ensure the data generation script has run and the path is correct.")
else:
    dropdown = widgets.Dropdown(
        options=scans,
        description='Select Scan:',
        style={'description_width': 'initial'},
        layout={'width': 'max-content'}
    )
    
    button = widgets.Button(
        description='Visualize Mesh',
        button_style='success',
        icon='eye'
    )
    
    status_output = widgets.Output()
    
    # Initialize a Plotly FigureWidget
    fig_widget = go.FigureWidget()
    fig_widget.layout = dict(
        scene=dict(
            xaxis=dict(visible=False), 
            yaxis=dict(visible=False), 
            zaxis=dict(visible=False), 
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        height=600
    )
    
    def on_button_clicked(b):
        with status_output:
            clear_output(wait=True)
            data = dropdown.value
            print(f"Loading mesh data... this may take a few seconds.\nFile: {data['obj']}")
            
        try:
            # Load mesh
            mesh = trimesh.load(data['obj'], process=False)
            
            # Load labels
            with open(data['json'], 'r') as f:
                label_data = json.load(f)
                
            labels = np.array(label_data.get('labels', np.zeros(len(mesh.vertices))))
            fault_labels = np.array(label_data.get('fault_labels', np.zeros(len(mesh.vertices))))
            
            # Assign colors
            colors = np.ones((len(mesh.vertices), 3)) * 255
            colors[labels == 0] = [230, 150, 160]   # Gum
            colors[labels > 0] = [240, 240, 240]    # Teeth
            colors[fault_labels == 1] = [255, 30, 30] # Fault
            
            vertex_colors = [f'rgb({int(c[0])}, {int(c[1])}, {int(c[2])})' for c in colors]
            
            # Create new trace
            trace = go.Mesh3d(
                x=mesh.vertices[:, 0], 
                y=mesh.vertices[:, 1], 
                z=mesh.vertices[:, 2],
                i=mesh.faces[:, 0], 
                j=mesh.faces[:, 1], 
                k=mesh.faces[:, 2],
                vertexcolor=vertex_colors, 
                showscale=False,
                lighting=dict(ambient=0.5, diffuse=0.8, specular=0.1, roughness=0.6)
            )
            
            # Update the widget safely
            with fig_widget.batch_update():
                fig_widget.data = []
                fig_widget.add_trace(trace)
                
            with status_output:
                clear_output(wait=True)
                print("Loaded successfully!")
                
        except Exception as e:
            with status_output:
                clear_output(wait=True)
                print(f"Error: {e}")
            
    button.on_click(on_button_clicked)
    
    # Display the UI with the FigureWidget explicitly included
    ui = widgets.VBox([widgets.HBox([dropdown, button]), status_output, fig_widget])
    display(ui)